In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120 
const aa = 40
const N  = 2_701_767
const I0 = 3
const S0 = 2_701_767 - 3 
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
tau = length(Istar_obs)

model_tag_sym = :reciprocal

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 3

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [reciprocal_model] Fitting chain 3 (tau=59)
[ Info: [reciprocal] iter 1000/1000000 elapsed=5.6s, rate=0.159, mean=[0.718, 0.00235, 0.745, 0.778], std=[0.0571, 0.000836, 0.1090, 0.0231] [ADAPT]
[ Info: [reciprocal] iter 2000/1000000 elapsed=10.3s, rate=0.128, mean=[0.776, 0.00173, 0.747, 0.777], std=[0.0663, 0.000817, 0.0773, 0.0176] [ADAPT]
[ Info: [reciprocal] iter 3000/1000000 elapsed=14.2s, rate=0.114, mean=[0.808, 0.00149, 0.750, 0.774], std=[0.0682, 0.000732, 0.0633, 0.0167] [ADAPT]
[ Info: [reciprocal] iter 4000/1000000 elapsed=18.1s, rate=0.109, mean=[0.819, 0.00138, 0.748, 0.772], std=[0.0627, 0.000661, 0.0556, 0.0179] [ADAPT]
[ Info: [reciprocal] iter 5000/1000000 elapsed=22.0s, rate=0.108, mean=[0.826, 0.00130, 0.744, 0.782], std=[0.0576, 0.000611, 0.0504, 0.0258] [ADAPT]
[ Info: [reciprocal] iter 6000/1000000 elapsed=25.9s, rate=0.102, mean=[0.829, 0.00125, 0.742, 0.785], std=[0.0532, 0.000568, 0.0463, 0.0246] [ADAPT]
[ Info: [reciprocal] iter 7000/1000000 elapsed=29